# Offline ALNS Repair Model Training (Improved)

This notebook runs the end-to-end offline training pipeline for the Hybrid ALNS
repair model with explicit reproducibility, dataset integrity checks, quality
gates, and covariate-shift mitigation.

Steps:
1. Generate synthetic data (v1)
2. Train baseline model (v1)
3. Collect ALNS repair states
4. Generate smaller synthetic data (v2) and retrain

## Colab setup (optional)

If running on Google Colab, upload the repo zip when prompted. The next
cell finds the repo root and installs dependencies.

In [ ]:
import os
import sys
import zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules


def find_repo_root(start: str) -> Path:
    for root, _, files in os.walk(start):
        if "pyproject.toml" in files and "requirements.txt" in files:
            return Path(root)
    raise FileNotFoundError("Could not find repo root with pyproject.toml")


repo_root = None
if IN_COLAB:
    try:
        repo_root = find_repo_root("/content")
    except FileNotFoundError:
        from google.colab import files

        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("Please upload the repo zip to continue.")
        zip_name = next(iter(uploaded))
        with zipfile.ZipFile(zip_name, "r") as zip_ref:
            zip_ref.extractall("/content")
        repo_root = find_repo_root("/content")
else:
    repo_root = find_repo_root(os.getcwd())

os.chdir(repo_root)
print("Repo root:", repo_root)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .

In [ ]:
import random
import numpy as np

SEED = 42
SYNTHETIC_V1_SEED = 0
SYNTHETIC_V2_SEED = 2
ALNS_SEED = 1

random.seed(SEED)
np.random.seed(SEED)

TRAINING_DIR = (
    Path(repo_root)
    / "bin_packing_optimization"
    / "hybrid_learning_metaheuristics"
    / "hybrid_alns"
    / "repair_model_training"
)
DATA_DIR = TRAINING_DIR / "training_data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(TRAINING_DIR)
print("Training dir:", TRAINING_DIR)

from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns.repair_model_training.generate_dataset import (
    GenerateDatasetConfig,
    generate_dataset,
)
from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns.repair_model_training.collect_alns_states import (
    CollectAlnsStatesConfig,
    collect_alns_states,
)
from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns.repair_model_training.train_repair_model import (
    TrainRepairModelConfig,
    train_repair_model,
)

## Step 1: Generate baseline synthetic data

In [ ]:
generate_dataset(
    GenerateDatasetConfig(
        instances=4000,
        n_min=50,
        n_max=200,
        max_negatives=5,
        seed=SYNTHETIC_V1_SEED,
        workers=1,
        output=str(DATA_DIR / "synthetic_v1.pkl"),
    )
)

## Step 2: Train baseline model (v1)

In [ ]:
train_repair_model(
    TrainRepairModelConfig(
        data=[str(DATA_DIR / "synthetic_v1.pkl")],
        output=str(TRAINING_DIR / "repair_model_v1.pkl"),
        seed=SEED,
        min_roc_auc=0.80,
        min_average_precision=0.60,
        require_alns_states=False,
        cv_folds=5,
        no_learning_curves=True,
        no_plots=True,
    )
)

## Step 3: Collect ALNS repair states

In [ ]:
collect_alns_states(
    CollectAlnsStatesConfig(
        model_path=str(TRAINING_DIR / "repair_model_v1.pkl"),
        instances=500,
        n_min=50,
        n_max=200,
        max_negatives=5,
        iterations=200,
        seed=ALNS_SEED,
        output=str(DATA_DIR / "alns_states_v1.pkl"),
    )
)

## Step 4: Retrain with ALNS states (v2)

In [ ]:
generate_dataset(
    GenerateDatasetConfig(
        instances=2000,
        n_min=50,
        n_max=200,
        max_negatives=3,
        seed=SYNTHETIC_V2_SEED,
        workers=1,
        output=str(DATA_DIR / "synthetic_v2.pkl"),
    )
)

train_repair_model(
    TrainRepairModelConfig(
        data=[
            str(DATA_DIR / "synthetic_v2.pkl"),
            str(DATA_DIR / "alns_states_v1.pkl"),
        ],
        output=str(TRAINING_DIR / "repair_model_v2.pkl"),
        seed=SEED,
        require_alns_states=True,   # v2: raises if no ALNS data present
        min_roc_auc=0.80,
        min_average_precision=0.60,
        cv_folds=3,
        no_learning_curves=True,
        no_plots=True,
    )
)

## Optional benchmark

In [ ]:
RUN_BENCHMARK = False
if RUN_BENCHMARK:
    from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns import (
        hybrid_alns_solver,
    )
    from bin_packing_optimization.utilities.benchmarking import create_benchmark

    benchmark = create_benchmark(
        dataset_key="falkenauer-u",
        solver_module=hybrid_alns_solver,
        time_limit=None,
    )
    benchmark.run(method=None, method_args={"max_iterations": 500})
    csv_path = benchmark.save_results_to_csv()
    print(csv_path)

## Validation checklist

After running all cells, verify the following in the printed output:

- [ ] **Reproducibility**: `config.seed`, `Python random seed`, and `numpy random seed` all print `42`
- [ ] **Integrity**: every dataset file shows `✓` with no NaN/inf and valid labels `[0, 1]`
- [ ] **Dataset summary**: merged-totals block shows rows, positive rate, and ALNS ratio
- [ ] **ALNS ratio > 0** for the v2 model (line `ALNS rows : ... (X.X% of total)`)
- [ ] **Quality gates**: both ROC-AUC and Average Precision print `✓` for v1 and v2 models
- [ ] **require_alns_states gate**: v1 training passes with `require_alns_states=False`; v2 would raise if ALNS file is removed
- [ ] **Model bundles saved**: `repair_model_v1.pkl` and `repair_model_v2.pkl` exist in `TRAINING_DIR`
